# 03 · Train the Qwen/Jev-style decision head on SageMaker

Experiment: a **frozen Qwen2.5-1.5B-Instruct backbone** supplies embeddings; only a three-class linear head is trained. There is no TypeSafe Jev API and no full-backbone fine-tuning. Checkpoints include train-only feature normalization, a separate calibration /temperature, and an exact resolved Qwen revision.


In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
## Import libs

In [3]:
from pathlib import Path
import os, sys, json

In [4]:
import random,hashlib,gc,platform
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import torch
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

### Utils

In [5]:
def clean(value):
    if isinstance(value, dict): return {str(k): clean(v) for k,v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)): return [clean(v) for v in value]
    if value is None or value is pd.NA: return None
    if isinstance(value, (bool, np.bool_)): return bool(value)
    if isinstance(value, (int, np.integer)): return int(value)
    if isinstance(value, (float, np.floating)): return float(value) if np.isfinite(value) else None
    if isinstance(value, (pd.Timestamp, datetime)): return utc(value).isoformat()
    return value

def canonical(value):
    return json.dumps(clean(value), sort_keys=True, separators=(",", ":"), allow_nan=False)

In [6]:
def utc(value):
    t = pd.Timestamp(value)
    if pd.isna(t):
        raise ValueError("A required timestamp is missing")
    return t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")

In [7]:
def softmax_np(logits,temperature=1.0):
    scores = np.asarray(logits,dtype=np.float64) / float(temperature)
    scores -= scores.max(axis=1,keepdims=True)
    ex = np.exp(scores)
    return ex/ex.sum(axis=1,keepdims=True)

In [8]:
def bool_value(value):
    if value is None or pd.isna(value): return None
    if isinstance(value, (bool, np.bool_)): return bool(value)
    if str(value).strip().lower() in {"true", "1", "1.0"}: return True
    if str(value).strip().lower() in {"false", "0", "0.0"}: return False
    raise ValueError(f"Invalid boolean: {value!r}")

In [9]:
def validate_data(data, allow_synthetic=True):
    df = data.copy()
    required = {"sample_id", "lake_id", "episode_id", "assessed_at", "features_available_at",
        "event_start_utc", "label_verified_until_utc", "label_status", "label_source", "is_synthetic"}
    missing = required - set(df.columns)
    if missing: raise ValueError(f"Missing metadata columns: {sorted(missing)}")
    if df.empty or df.sample_id.isna().any() or df.sample_id.duplicated().any():
        raise ValueError("Need nonempty data with unique sample_id values")
    for col in ["lake_id", "episode_id", "label_source"]:
        if df[col].isna().any() or df[col].astype(str).str.strip().eq("").any():
            raise ValueError(f"Missing {col}")
    df["is_synthetic"] = df.is_synthetic.map(bool_value)
    if df.is_synthetic.isna().any(): raise ValueError("is_synthetic must be explicitly true/false")
    if df.is_synthetic.any() and not allow_synthetic:
        raise ValueError("Synthetic rows are forbidden in real training mode")
    expected_status = np.where(df.is_synthetic, "SYNTHETIC", "VERIFIED")
    if not (df.label_status.to_numpy() == expected_status).all():
        raise ValueError("Unverified labels: complete independent labelling before training")
    if df.is_synthetic.any() and not df.is_synthetic.all():
        raise ValueError("Keep synthetic demonstrations and real event training separate")
    for col in FEATURE_COLUMNS:
        if col not in df: df[col] = np.nan
        if col == "possible_ice_cover":
            df[col] = df[col].map(bool_value).map({True:1.0, False:0.0})
        else:
            df[col] = pd.to_numeric(df[col], errors="raise")
        if np.isinf(df[col].to_numpy(float)).any(): raise ValueError(f"Infinite values in {col}")
    for col in ["area_km2", "baseline_km2", "optical_age_days", "sar_age_days", "days_since_prev",
        "n_history", "sar_n_baseline", "eq_nearest_km_24h", "gpm_rain_24h_mm",
        "wx_rain_24h_mm", "wx_rain_72h_mm", "wx_forecast_rain_24h_mm", "wx_forecast_rain_72h_mm"]:
        if (df[col].dropna() < 0).any(): raise ValueError(f"Negative values in {col}")
    if not df.footprint_valid_frac.dropna().between(0,1).all():
        raise ValueError("footprint_valid_frac must be between zero and one")
    df["target_id"] = [derive_target(r) for r in df.to_dict("records")]
    if "event_class" in df and df.event_class.notna().any():
        expected = df.target_id.map(dict(enumerate(CLASS_NAMES)))
        if not df.event_class.fillna(expected).eq(expected).all():
            raise ValueError("Provided event_class disagrees with onset/coverage timestamps")
    if set(df.target_id) != {0,1,2}: raise ValueError("This training task needs examples of all three outcomes")
    df["target_name"] = df.target_id.map(dict(enumerate(CLASS_NAMES)))
    return df

In [10]:
def split_dataset(df, mode="chronological", seed=42):
    result = df.copy()
    names = ("train", "validation", "calibration", "test")
    if mode == "group":
        groups = np.array(sorted(result.lake_id.astype(str).unique()))
        if len(groups) < 20: raise ValueError("Need at least 20 lake groups for this four-way example split")
        np.random.default_rng(seed).shuffle(groups)
        cuts = [int(len(groups)*.60), int(len(groups)*.75), int(len(groups)*.85)]
        assignment = {g:name for name,chunk in zip(names,np.split(groups,cuts)) for g in chunk}
        result["split"] = result.lake_id.astype(str).map(assignment)
    elif mode == "chronological":
        ts = pd.to_datetime(result.assessed_at, utc=True)
        times = np.array(sorted(ts.unique()))
        if len(times) < 8: raise ValueError("Not enough distinct clocks for chronological splitting")
        cuts = [times[int(len(times)*fraction)] for fraction in (.60,.75,.85)]
        result["split"] = np.select([ts<cuts[0],ts<cuts[1],ts<cuts[2]],names[:3],default="test")
        keep = np.ones(len(result),dtype=bool)
        for name,boundary in zip(names[:3],cuts):
            keep &= ~((result.split == name) & (ts + pd.Timedelta(hours=72) >= boundary))
        print("Purged horizon-boundary rows:",int((~keep).sum()))
        result = result.loc[keep].copy()
    elif mode == "provided":
        if "split" not in result: raise ValueError("Supply a split column")
        if not result.split.isin(names).all(): raise ValueError("Unexpected split names")
    else:
        raise ValueError("Unknown SPLIT_MODE")
    if result.groupby("episode_id").split.nunique().max() > 1:
        raise ValueError("An episode crosses splits; assign whole event/control episodes together")
    if mode != "chronological" and result.groupby("lake_id").split.nunique().max() > 1:
        raise ValueError("Lake leakage across splits")
    for name in names:
        subset = result[result.split == name]
        if set(subset.target_id) != {0,1,2}:
            raise ValueError(f"{name} lacks a target class; collect more independent episodes")
    return result.reset_index(drop=True)

In [11]:
def feature_prompt(row):
    features = {}
    for name in FEATURE_COLUMNS:
        value = row.get(name)
        if name == "possible_ice_cover":
            features[name] = bool_value(value)
        else:
            try: number = float(value)
            except (TypeError,ValueError): number = np.nan
            features[name] = round(number,4) if np.isfinite(number) else None
    return ("Glacial lake measurements available now. Predict the FIRST new outburst onset in "
        "the next 72 hours. Units: area km2, change percent, rain mm, temperature C, age days, "
        "distance km. Null=unknown. Optical and SAR use separate baselines. Shrinkage may be "
        "seasonal or past drainage; future onset is a different target.\nFEATURES=" + canonical(features) +
        "\nDECISION FEATURES:")

In [12]:
# UPDATED: reproducible balanced synthetic demo generator; no real lake data used.
GENERATOR_VERSION = "GLOFWATCH_TOY_BALANCED_V2"
CLASS_NAMES = ("NO_ONSET_WITHIN_72H", "ONSET_0_TO_24H", "ONSET_24_TO_72H")
FEATURE_COLUMNS = [
    "area_km2", "baseline_km2", "pct_vs_baseline", "pct_vs_prev", "days_since_prev", "z",
    "n_history", "footprint_valid_frac", "optical_age_days", "sar_pct_change", "sar_z",
    "sar_n_baseline", "sar_age_days", "wx_rain_24h_mm", "wx_rain_72h_mm",
    "wx_forecast_rain_24h_mm", "wx_forecast_rain_72h_mm", "wx_temp_c", "wx_temp_max_72h_c",
    "eq_max_mag_near_72h", "eq_nearest_km_24h", "gpm_rain_24h_mm", "possible_ice_cover",
]
METADATA_COLUMNS = [
    "sample_id", "lake_id", "episode_id", "assessed_at", "features_available_at",
    "event_start_utc", "label_verified_until_utc", "label_status", "label_source",
    "is_synthetic", "event_class",
]


def make_synthetic_sample(seed=42, n_lakes=60, episodes_per_lake=60):
    """Return n_lakes * episodes_per_lake rows, exactly one third per class.

    Features are drawn first and a noisy toy outcome is sampled from them.
    Rejection sampling balances outcomes within every fictional lake. This
    changes prevalence and cannot yield real-world occurrence probabilities.
    Each retained row represents a separate episode, eight days apart.
    """
    if n_lakes < 20:
        raise ValueError("Use at least 20 fictional lakes for the four-way group split")
    if episodes_per_lake < 3 or episodes_per_lake % 3:
        raise ValueError("episodes_per_lake must be a positive multiple of 3")
    rng = np.random.default_rng(seed)
    quota = episodes_per_lake // 3
    rows = []
    for lake in range(n_lakes):
        lake_base = float(np.exp(rng.uniform(np.log(.03), np.log(2.0))))
        hidden_fragility = float(rng.normal(0, .45))
        counts = np.zeros(3, dtype=int)
        episode, attempts = 0, 0
        while (counts < quota).any():
            attempts += 1
            if attempts > episodes_per_lake * 1000:
                raise RuntimeError("Could not fill all simulated classes for this lake")
            base = round(lake_base * float(rng.uniform(.9, 1.1)), 6)
            pct = round(float(np.clip(rng.normal(5, 18), -60, 85)), 4)
            previous = float(np.clip(.35 * pct + rng.normal(0, 9), -60, 70))
            rain24 = float(rng.gamma(1.2, 14))
            rain72 = rain24 + float(rng.gamma(1.5, 20))
            forecast24 = float(rng.gamma(1.2, 20))
            forecast72 = forecast24 + float(rng.gamma(1.5, 23))
            temp = float(np.clip(rng.normal(6, 6), -12, 18))
            sarpct = float(np.clip(pct + rng.normal(0, 12), -80, 100))

            # Arbitrary, noisy toy relationships: never use as operational rules.
            latent = (-2.7 + .04 * max(pct, 0) + .022 * forecast72
                      + .04 * max(temp, 0) + hidden_fragility + rng.normal(0, 1.0))
            onset_probability = 1 / (1 + np.exp(-np.clip(latent, -20, 20)))
            target = 0
            if rng.random() < onset_probability:
                urgency = (-.4 + .022 * forecast24 - .009 * (forecast72 - forecast24)
                           + .015 * max(previous, 0) + rng.normal(0, .7))
                soon_probability = 1 / (1 + np.exp(-np.clip(urgency, -20, 20)))
                target = 1 if rng.random() < soon_probability else 2
            if counts[target] >= quota:
                continue

            at = pd.Timestamp("2024-06-01T00:00:00Z") + pd.Timedelta(days=8 * episode + lake % 5)
            onset = None
            if target:
                # Integer seconds keep the (0,24]h and (24,72]h boundaries exact.
                seconds = int(rng.integers(1, 86401) if target == 1 else rng.integers(86401, 259201))
                onset = (at + pd.Timedelta(seconds=seconds)).isoformat()
            eq_seen = bool(rng.random() < .12)
            row = {
                "area_km2": round(base * (1 + pct / 100), 6),
                "baseline_km2": base, "pct_vs_baseline": pct, "pct_vs_prev": previous,
                "days_since_prev": float(rng.uniform(4, 12)), "z": pct / float(rng.uniform(5, 12)),
                "n_history": int(rng.integers(4, 40)), "footprint_valid_frac": float(rng.uniform(.75, 1)),
                "optical_age_days": float(rng.uniform(2, 14)), "sar_pct_change": sarpct,
                "sar_z": sarpct / float(rng.uniform(7, 16)), "sar_n_baseline": int(rng.integers(4, 20)),
                "sar_age_days": float(rng.uniform(2, 10)), "wx_rain_24h_mm": rain24,
                "wx_rain_72h_mm": rain72, "wx_forecast_rain_24h_mm": forecast24,
                "wx_forecast_rain_72h_mm": forecast72, "wx_temp_c": temp,
                "wx_temp_max_72h_c": temp + float(rng.uniform(0, 5)),
                "eq_max_mag_near_72h": float(rng.uniform(2.5, 5.5)) if eq_seen else None,
                "eq_nearest_km_24h": float(rng.uniform(15, 230)) if eq_seen else None,
                "gpm_rain_24h_mm": max(0, rain24 + float(rng.normal(0, 5))),
                "possible_ice_cover": temp < -2,
                "sample_id": f"SYNTH_L{lake:03d}_E{episode:04d}", "lake_id": f"SYNTH_L{lake:03d}",
                "episode_id": f"SYNTH_L{lake:03d}_E{episode:04d}", "assessed_at": at.isoformat(),
                "features_available_at": (at - pd.Timedelta(minutes=int(rng.integers(0, 121)))).isoformat(),
                "event_start_utc": onset,
                "label_verified_until_utc": (at + pd.Timedelta(hours=96)).isoformat(),
                "label_status": "SYNTHETIC", "label_source": f"{GENERATOR_VERSION}; seed={seed}; NOT_REAL",
                "is_synthetic": True, "event_class": CLASS_NAMES[target],
            }
            # Missing context stays unknown. SAR remains usable for every row.
            if rng.random() < .08:
                row["gpm_rain_24h_mm"] = None
            if rng.random() < .06:
                row["wx_forecast_rain_24h_mm"] = row["wx_forecast_rain_72h_mm"] = None
            if rng.random() < .08:
                for key in FEATURE_COLUMNS[:9]:
                    row[key] = None
            for key in FEATURE_COLUMNS:
                value = row[key]
                if isinstance(value, (float, np.floating)) and key not in {"area_km2", "baseline_km2"}:
                    row[key] = round(float(value), 4)
            rows.append(row)
            counts[target] += 1
            episode += 1
    return pd.DataFrame(rows, columns=FEATURE_COLUMNS + METADATA_COLUMNS)


In [13]:
def fit_temperature(logits,labels):
    candidates = np.geomspace(.2,5.0,161)
    losses = [log_loss(labels,softmax_np(logits,t),labels=[0,1,2]) for t in candidates]
    best = int(np.argmin(losses))
    return float(candidates[best]),float(losses[best])

In [14]:
def measurement_gate(row,now):
    warnings = []
    try:
        at = utc(row.get("assessed_at"))
        available = utc(row.get("features_available_at"))
    except (ValueError, TypeError):
        return ["Missing or invalid assessment/availability timestamp"]
    if available > at:
        warnings.append("Features are future-dated")
    age_h = (utc(now)-at).total_seconds()/3600
    if age_h < 0 or age_h > 12: warnings.append("Assessment missing the current 12-hour freshness window")
    if row.get("features_available_at") is not None and pd.notna(row.get("features_available_at")):
        if utc(row["features_available_at"]) > at: warnings.append("Features are future-dated")
    def num(key):
        try: x = float(row.get(key)); return x if np.isfinite(x) else None
        except (ValueError,TypeError): return None
    oa,sa = num("optical_age_days"),num("sar_age_days")
    coverage,n = num("footprint_valid_frac"),num("n_history")
    opt_ok = (oa is not None and 0<=oa<=15 and coverage is not None and 0.8<=coverage<=1.0 and
        n is not None and n>=3 and (num("baseline_km2") or 0)>0 and num("pct_vs_baseline") is not None)
    sar_ok = (sa is not None and 0<=sa<=12 and (num("sar_n_baseline") or 0)>=3 and
        num("sar_pct_change") is not None and num("sar_z") is not None)
    if not opt_ok and not sar_ok: warnings.append("No usable recent lake-change measurement")
    return warnings

In [15]:
def derive_target(row):
    at, available = utc(row["assessed_at"]), utc(row["features_available_at"])
    if available > at:
        raise ValueError(f"{row['sample_id']}: features were not available at the prediction clock")
    event = row.get("event_start_utc")
    if pd.notna(event) and str(event).strip():
        delta = (utc(event) - at).total_seconds() / 3600
        if delta <= 0:
            raise ValueError(f"{row['sample_id']}: post-onset row; use a separate detection dataset")
        if delta <= 24: return 1
        if delta <= 72: return 2
    end = row.get("label_verified_until_utc")
    if pd.isna(end) or utc(end) < at + pd.Timedelta(hours=72):
        raise ValueError(f"{row['sample_id']}: cannot infer a negative without verified 72-hour coverage")
    return 0

### Feature Details

In [16]:
"""Measurement contract and label validation. No TensorFlow/PyTorch imports.
"""
import json
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

In [17]:
PROMPT_VERSION = "glof-features-v1"
CLASS_NAMES = ("NO_ONSET_WITHIN_72H", "ONSET_0_TO_24H", "ONSET_24_TO_72H")
FEATURE_COLUMNS = [
    "area_km2", "baseline_km2", "pct_vs_baseline", "pct_vs_prev", "days_since_prev", "z",
    "n_history", "footprint_valid_frac", "optical_age_days",
    "sar_pct_change", "sar_z", "sar_n_baseline", "sar_age_days",
    "wx_rain_24h_mm", "wx_rain_72h_mm", "wx_forecast_rain_24h_mm", "wx_forecast_rain_72h_mm",
    "wx_temp_c", "wx_temp_max_72h_c", "eq_max_mag_near_72h", "eq_nearest_km_24h",
    "gpm_rain_24h_mm", "possible_ice_cover"
]

### Qwen Jev Model

In [18]:
"""Frozen Qwen backbone + trainable three-class Jev-style head.
"""
import copy
import json
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoConfig, AutoModel, AutoTokenizer
from sklearn.metrics import (log_loss, brier_score_loss, average_precision_score,
                             roc_auc_score, confusion_matrix)

In [19]:
class GLOFJev(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        for parameter in backbone.parameters(): parameter.requires_grad = False
        self.score = nn.Linear(backbone.config.hidden_size, 3)
        self.register_buffer("feature_mean",torch.zeros(backbone.config.hidden_size))
        self.register_buffer("feature_scale",torch.ones(backbone.config.hidden_size))

    def features(self, input_ids, attention_mask):
        if (attention_mask.sum(1) == 0).any(): raise ValueError("Empty token sequence")
        self.backbone.eval()
        with torch.no_grad():
            hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
                use_cache=False, return_dict=True).last_hidden_state
            # Last mask=1 position supports either padding side; tokenizer uses right padding.
            indexes = torch.arange(attention_mask.shape[1],device=attention_mask.device)
            last = (indexes.unsqueeze(0)*attention_mask).max(dim=1).values.long()
            pooled = hidden[torch.arange(hidden.shape[0],device=hidden.device),last]
        return pooled.float()

    def forward(self,input_ids,attention_mask):
        pooled = self.features(input_ids,attention_mask)
        return self.score((pooled-self.feature_mean)/self.feature_scale)

In [20]:
def build_backbone():
    cfg = AutoConfig.from_pretrained(MODEL_ID,revision=MODEL_REVISION,trust_remote_code=False)
    resolved = getattr(cfg,"_commit_hash",None) or MODEL_REVISION or "main"
    if hasattr(cfg,"sliding_window"): cfg.sliding_window = None
    tok = AutoTokenizer.from_pretrained(MODEL_ID,revision=resolved,trust_remote_code=False)
    tok.padding_side = "right"
    tok.pad_token = tok.pad_token or tok.eos_token
    backbone = AutoModel.from_pretrained(MODEL_ID,revision=resolved,config=cfg,
        torch_dtype=BACKBONE_DTYPE,trust_remote_code=False,attn_implementation="sdpa")
    wrapped = GLOFJev(backbone).to(DEVICE)
    assert all(not p.requires_grad for p in wrapped.backbone.parameters())
    return tok,wrapped,resolved

In [21]:
def encode_no_truncation(tokenizer,texts,device):
    raw = tokenizer(texts,add_special_tokens=True,truncation=False)["input_ids"]
    longest = max(map(len,raw))
    if longest > MAX_LENGTH:
        raise ValueError(f"Prompt length {longest} exceeds MAX_LENGTH={MAX_LENGTH}; increase it rather than truncating features")
    batch = tokenizer(texts,padding=True,truncation=False,return_tensors="pt")
    return {key:batch[key].to(device) for key in ("input_ids","attention_mask")}

def extract_embeddings(model,tokenizer,texts,batch_size=2):
    vectors = []
    model.eval()
    device = model.score.weight.device
    for start in range(0,len(texts),batch_size):
        batch = encode_no_truncation(tokenizer,texts[start:start+batch_size],device)
        vectors.append(model.features(**batch).detach().cpu().numpy().astype(np.float32))
        if start == 0 or start+batch_size >= len(texts) or start % 100 == 0:
            print(f"Qwen features: {min(start+batch_size,len(texts))}/{len(texts)}")
    return np.concatenate(vectors)

In [22]:
def fit_head(embeddings,table,epochs=60):
    train_mask = table.split.eq("train").to_numpy()
    val_mask = table.split.eq("validation").to_numpy()
    mean = embeddings[train_mask].mean(axis=0).astype(np.float32)
    scale = np.maximum(embeddings[train_mask].std(axis=0),1e-5).astype(np.float32)
    x = torch.from_numpy(((embeddings-mean)/scale).astype(np.float32))
    y = torch.tensor(table.target_id.to_numpy(),dtype=torch.long)
    torch.manual_seed(SEED)
    head = nn.Linear(embeddings.shape[1],3)
    optimizer = torch.optim.AdamW(head.parameters(),lr=HEAD_LR,weight_decay=HEAD_WEIGHT_DECAY)
    loader = DataLoader(TensorDataset(x[train_mask],y[train_mask]),batch_size=HEAD_BATCH_SIZE,
        shuffle=True,generator=torch.Generator().manual_seed(SEED))
    best,stale,history = float("inf"),0,[]
    best_state = copy.deepcopy(head.state_dict())
    for epoch in range(1,epochs+1):
        head.train(); running,count = 0.,0
        for bx,by in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(head(bx),by)
            loss.backward(); optimizer.step()
            running += loss.item()*len(by); count += len(by)
        head.eval()
        with torch.no_grad():
            val_loss = nn.functional.cross_entropy(head(x[val_mask]),y[val_mask]).item()
        history.append({"epoch":epoch,"train_nll":running/count,"validation_nll":val_loss})
        if val_loss < best - 1e-5:
            best,best_state,stale = val_loss,copy.deepcopy(head.state_dict()),0
        else: stale += 1
        if epoch == 1 or epoch%5 == 0: print(f"epoch {epoch}: train NLL={running/count:.4f}, validation NLL={val_loss:.4f}")
        if stale >= EARLY_STOP_PATIENCE:
            print("Early stop at epoch",epoch); break
    head.load_state_dict(best_state); head.eval()
    with torch.no_grad(): logits = head(x).numpy()
    return head,mean,scale,logits,pd.DataFrame(history)

In [23]:
def metrics_report(y,probabilities,threshold=.5):
    p = np.asarray(probabilities)
    if p.shape != (len(y),3) or not np.isfinite(p).all() or not np.allclose(p.sum(1),1):
        raise ValueError("Invalid class probabilities")
    result = {"multiclass_nll":float(log_loss(y,p,labels=[0,1,2])),"rows":len(y)}
    for horizon in (24,72):
        truth = (y==1).astype(int) if horizon==24 else (y>0).astype(int)
        score = p[:,1] if horizon==24 else p[:,1]+p[:,2]
        tn,fp,fn,tp = confusion_matrix(truth,score>=threshold,labels=[0,1]).ravel()
        result[str(horizon)+"h"] = {"brier":float(brier_score_loss(truth,score)),
            "prevalence":float(truth.mean()),
            "average_precision":float(average_precision_score(truth,score)) if truth.sum() else None,
            "roc_auc":float(roc_auc_score(truth,score)) if len(set(truth))==2 else None,
            "recall":float(tp/(tp+fn)) if tp+fn else None,
            "false_positive_rate":float(fp/(fp+tn)) if fp+tn else None,
            "tp":int(tp),"fp":int(fp),"fn":int(fn),"tn":int(tn),"threshold":threshold}
    return result

In [24]:
def load_trained_head(artifact_dir):
    global MAX_LENGTH
    artifact_dir = Path(artifact_dir)
    meta = json.loads((artifact_dir/"glof_jev_metadata.json").read_text())
    if not (0 < float(meta.get("temperature", 0)) < 100):
        raise ValueError("Invalid saved temperature")
    if meta["feature_columns"] != FEATURE_COLUMNS or tuple(meta["class_names"]) != CLASS_NAMES or meta["prompt_version"] != PROMPT_VERSION:
        raise ValueError("Checkpoint schema/prompt differs from this notebook")
    MAX_LENGTH = int(meta["max_length"])
    if MAX_LENGTH < 1:
        raise ValueError("Invalid saved token limit")
    model_source = meta["model_id"]
    revision = meta["model_revision"]
    if (artifact_dir/"backbone"/"config.json").is_file():
        identity = json.loads((artifact_dir/"backbone_identity.json").read_text())
        if identity != {"model_id":model_source,"revision":revision}:
            raise ValueError("Local Qwen backbone does not match saved head identity")
        model_source,revision = str(artifact_dir/"backbone"),None
    cfg = AutoConfig.from_pretrained(model_source,revision=revision,trust_remote_code=False)
    if hasattr(cfg,"sliding_window"): cfg.sliding_window = None
    tok = AutoTokenizer.from_pretrained(model_source,revision=revision,trust_remote_code=False)
    tok.padding_side = "right"; tok.pad_token = tok.pad_token or tok.eos_token
    backbone = AutoModel.from_pretrained(model_source,revision=revision,config=cfg,
        torch_dtype=BACKBONE_DTYPE,trust_remote_code=False,attn_implementation="sdpa")
    model = GLOFJev(backbone).to(DEVICE)
    saved = torch.load(artifact_dir/"glof_jev_head.pt",map_location=DEVICE,weights_only=True)
    model.score.load_state_dict(saved["score_state"])
    if not torch.isfinite(saved["feature_scale"]).all() or (saved["feature_scale"] <= 0).any():
        raise ValueError("Invalid saved feature normalization")
    model.feature_mean.copy_(saved["feature_mean"]); model.feature_scale.copy_(saved["feature_scale"])
    model.eval()
    return tok,model,meta

In [25]:
def predict_glof(rows,tokenizer,model,metadata,now=None):
    now = utc(now or datetime.now(timezone.utc))
    results = []
    for row in rows:
        warnings = measurement_gate(row,now)
        result = {"lake_id":row.get("lake_id"),"assessed_at":row.get("assessed_at"),
            "synthetic_demo_only":bool(metadata["synthetic_demo_only"]),"warnings":warnings,"features_available_at":row.get("features_available_at"),
            "answers":None,"status":"INSUFFICIENT_DATA"}
        if not warnings:
            batch = encode_no_truncation(tokenizer,[feature_prompt(row)],model.score.weight.device)
            with torch.no_grad(): raw = model(**batch).detach().cpu().numpy()
            p = softmax_np(raw,metadata["temperature"])[0]
            choice = CLASS_NAMES[int(p.argmax())]
            result["answers"] = {
                "glof_next_24h":{"type":"noul","noul":float(p[1])},
                "glof_next_72h":{"type":"noul","noul":float(p[1]+p[2])},
                "onset_window":{"type":"choice","choice":choice,
                    "probabilities":dict(zip(CLASS_NAMES,map(float,p)))}}
            result["status"] = "SYNTHETIC_DEMO_ONLY" if metadata["synthetic_demo_only"] else "RESEARCH_MODEL_OUTPUT"
            result["requires_review"] = bool(p[1]+p[2]>=RESEARCH_THRESHOLD_72H)
        results.append(clean(result))
    return results

### Run

In [26]:
SEED = 42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = None
MAX_LENGTH = 768
HEAD_LR = 1e-3
HEAD_WEIGHT_DECAY = 1e-2
HEAD_BATCH_SIZE = 64
EARLY_STOP_PATIENCE = 20
RESEARCH_THRESHOLD_72H = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BACKBONE_DTYPE = (torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
                  else torch.float16 if DEVICE.type == "cuda" else torch.float32)

In [27]:
pwd

'/content'

In [28]:
# UPDATED: use the 3,600-row synthetic demo instead of the older 720-row file.
USE_SYNTHETIC_SAMPLE = True
DEFAULT_DATA_CSV = (Path("/content/jev_training_SYNTHETIC.csv")
                    if "google.colab" in sys.modules else Path("jev_training_SYNTHETIC.csv"))
# Change DATA_CSV to your uploaded CSV's full path if it is in another folder.
DATA_CSV = Path(os.environ.get("GLOF_JEV_TRAINING_CSV", str(DEFAULT_DATA_CSV))).expanduser()
SYNTHETIC_LAKES = 60
SYNTHETIC_EPISODES_PER_LAKE = 60   # divisible by 3; rows = lakes * episodes
REGENERATE_SYNTHETIC_DATA = False # True regenerates this CSV using the controls above


In [29]:
# UPDATED: save this run in its own artifact subfolder.
DEFAULT_ARTIFACT_BASE = (Path("/content/drive/MyDrive/Codes")
                         if Path("/content/drive/MyDrive").is_dir() else Path.cwd() / "artifacts")
ARTIFACT_BASE = Path(os.environ.get("GLOF_JEV_ARTIFACT_BASE", str(DEFAULT_ARTIFACT_BASE))).expanduser()
ARTIFACT_DIR = ARTIFACT_BASE / ("jev_synthetic_demo" if USE_SYNTHETIC_SAMPLE else "jev_verified_training")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
(ARTIFACT_DIR / "data").mkdir(parents=True, exist_ok=True)
print("Artifacts:", ARTIFACT_DIR.resolve())


Artifacts: /content/artifacts/jev_synthetic_demo


### 2. Dataset contract and target definition

Each row is a feature snapshot **available at** `assessed_at`. `lake_id`, `episode_id`, dates, label provenance and future outcomes are for splitting/audit only; they never enter Qwen's prompt or the numeric baseline.

| Target class | Verified first event onset relative to assessed_at |
|---|---|
| `NO_ONSET_WITHIN_72H` | No new onset in `(0,72]` hours; the entire window must be verified event-free |
| `ONSET_0_TO_24H` | A new onset in `(0,24]` hours |
| `ONSET_24_TO_72H` | A new onset in `(24,72]` hours |

One softmax distribution gives `P(next24h)=P(class1)` and `P(next72h)=P(class1)+P(class2)`, so the horizons are always consistent. This predicts occurrence within a window, not flood magnitude, exact onset time or downstream inundation.

Required metadata: `sample_id`, `lake_id`, `episode_id`, `assessed_at`, `features_available_at`, `event_start_utc` (blank for no known event), `label_verified_until_utc`, `label_status`, `label_source`, and `is_synthetic`. Optional `event_class` is checked against the independently derived target. For real data `label_status=VERIFIED`; for this toy sample it is `SYNTHETIC`. A positive label needs a verified onset; a negative needs verified coverage through the full 72 hours. The timestamp field must describe the first onset in the monitored episode, not an arbitrary later flood.

Feature units: area km², rainfall mm, temperature °C, earthquake distance km, observation age days, coverage fraction 0–1. Percentage change is signed: negative = shrinkage. Radar and optical changes have separate baselines. Missing input values remain unknown.


In [30]:
# UPDATED: load the provided synthetic CSV, or generate the same 3,600-row demo.
if REGENERATE_SYNTHETIC_DATA or not DATA_CSV.is_file():
    if not USE_SYNTHETIC_SAMPLE:
        raise FileNotFoundError("Provide your independently verified real training CSV")
    DATA_CSV.parent.mkdir(parents=True, exist_ok=True)
    generated = make_synthetic_sample(seed=SEED, n_lakes=SYNTHETIC_LAKES,
                                      episodes_per_lake=SYNTHETIC_EPISODES_PER_LAKE)
    generated.to_csv(DATA_CSV, index=False)
    print("Generated explicitly synthetic training data:", DATA_CSV)

data = validate_data(pd.read_csv(DATA_CSV), allow_synthetic=USE_SYNTHETIC_SAMPLE)
IS_SYNTHETIC_RUN = bool(data.is_synthetic.all())
if USE_SYNTHETIC_SAMPLE and not IS_SYNTHETIC_RUN:
    raise ValueError("Synthetic demo mode requires an entirely synthetic table")
print("CSV:", DATA_CSV.resolve())
print("Rows:", len(data), "| independent lake IDs:", data.lake_id.nunique(),
      "| mode:", "SYNTHETIC_DEMO_ONLY" if IS_SYNTHETIC_RUN else "VERIFIED_RESEARCH_DATA")
display(data.target_name.value_counts())


CSV: /content/jev_training_SYNTHETIC.csv
Rows: 3600 | independent lake IDs: 60 | mode: SYNTHETIC_DEMO_ONLY


,count
target_name,
NO_ONSET_WITHIN_72H,1200
ONSET_0_TO_24H,1200
ONSET_24_TO_72H,1200


In [31]:
# UPDATED: separate prompt-dependent embedding caches from the saved model.
RUN_DIR = ARTIFACT_DIR / "embedding_cache"
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [32]:
# UPDATED: hold out whole fictional lakes for this balanced demo.
MAX_LENGTH = 768
EXTRACT_BATCH_SIZE = 16
HEAD_EPOCHS = 60
SPLIT_MODE = "group"
RESEARCH_THRESHOLD_72H = 0.50

In [33]:
DEVICE,BACKBONE_DTYPE

(device(type='cuda'), torch.bfloat16)

In [34]:
# Release a previous extraction model before rebuilding in the same kernel.
if "feature_model" in globals():
    del feature_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
tokenizer, feature_model, RESOLVED_MODEL_REVISION = build_backbone()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [35]:
# qj.MODEL_ID,qj.MODEL_REVISION,qj.MAX_LENGTH,qj.SEED = MODEL_ID,MODEL_REVISION,MAX_LENGTH,SEED
# DEVICE,BACKBONE_DTYPE = qj.DEVICE,qj.BACKBONE_DTYPE
# build_backbone,extract_embeddings,fit_head = qj.build_backbone,qj.extract_embeddings,qj.fit_head
# metrics_report,load_trained_head,predict_glof = qj.metrics_report,qj.load_trained_head,qj.predict_glof
print("Device:",DEVICE,"| dtype:",BACKBONE_DTYPE,"| synthetic training:",USE_SYNTHETIC_SAMPLE)

Device: cuda | dtype: torch.bfloat16 | synthetic training: True


#### 4. Split before fitting
Default: hold out lake IDs in approximately **60% train / 15% validation / 10% calibration / 15% test**. Validation selects the head; calibration fits one temperature; test is inspected only after those choices. No episode can cross a split. A lake-group split measures generalization to held-out lakes; it does not by itself prove performance on future years.

For real forecasting experiments, select `chronological` or supply a reviewed `split` column with these four names. The chronological option removes each block's rows whose next 72 hours cross into the next block and rejects episodes crossing remaining splits. If event-group or temporal checks fail, revise your split manifest rather than weakening the assertions. A supplied manifest must also keep lakes disjoint here; use an explicitly designed chronological experiment if you need the same lake over future periods.


In [36]:
data = split_dataset(data, SPLIT_MODE, SEED)
display(pd.crosstab(data.split, data.target_name))
data[["sample_id", "lake_id", "episode_id", "assessed_at", "split"]].to_csv(ARTIFACT_DIR /"data"/"split_manifest.csv",index=False)


target_name,NO_ONSET_WITHIN_72H,ONSET_0_TO_24H,ONSET_24_TO_72H
split,,,
calibration,120,120,120
test,180,180,180
train,720,720,720
validation,180,180,180


#### 5. Serialize only allowed input features
The same function is used for training and inference. Labels, lake names, coordinates, IDs, event dates and existing alert scores are deliberately absent. Numeric values are compact and missing values are explicit. Qwen is tested as an embedding backbone; there is no assumption that its pretrained language understanding supplies glaciological expertise.


In [37]:
prompts = [feature_prompt(r) for r in data.to_dict("records")]
for prompt in prompts:
    assert all(key not in prompt for key in ["event_start_utc", "sample_id", "lake_id", "episode_id", "event_class", "label_source"])
print(prompts[0])


Glacial lake measurements available now. Predict the FIRST new outburst onset in the next 72 hours. Units: area km2, change percent, rain mm, temperature C, age days, distance km. Null=unknown. Optical and SAR use separate baselines. Shrinkage may be seasonal or past drainage; future onset is a different target.
FEATURES={"area_km2":1.0114,"baseline_km2":0.8295,"days_since_prev":10.621,"eq_max_mag_near_72h":3.9002,"eq_nearest_km_24h":24.4178,"footprint_valid_frac":0.8386,"gpm_rain_24h_mm":3.9086,"n_history":9.0,"optical_age_days":13.6484,"pct_vs_baseline":21.9302,"pct_vs_prev":-9.8837,"possible_ice_cover":false,"sar_age_days":8.2271,"sar_n_baseline":16.0,"sar_pct_change":27.5403,"sar_z":1.8314,"wx_forecast_rain_24h_mm":5.8078,"wx_forecast_rain_72h_mm":56.9728,"wx_rain_24h_mm":1.8449,"wx_rain_72h_mm":18.9917,"wx_temp_c":12.7634,"wx_temp_max_72h_c":13.7366,"z":2.3276}
DECISION FEATURES:


#### 6. Frozen Qwen backbone and the small head
Compared with the tutorial, the head has three logits, pooling rejects empty masks, device dtype adapts to T4 versus newer GPUs, and inputs are never silently truncated. Feature extraction is cached once; the GPU backbone is freed before the cheap CPU head-training epochs.


In [38]:
num_head = sum(p.numel() for p in feature_model.score.parameters())
print("Trainable head parameters:",num_head, "| frozen backbone:",sum(p.numel() for p in feature_model.backbone.parameters()))
lengths = [len(ids) for ids in tokenizer(prompts, truncation=False)["input_ids"]]

Trainable head parameters: 4611 | frozen backbone: 1543714304


In [39]:
print("Prompt tokens median/max:",int(np.median(lengths)),max(lengths))
if max(lengths) > MAX_LENGTH: raise ValueError("Increase MAX_LENGTH and rerun before extraction")

Prompt tokens median/max: 363 381


In [40]:
# UPDATED: the cache identity changes when the model or any input prompt changes.
cache_meta = {"model_id": MODEL_ID, "revision": RESOLVED_MODEL_REVISION,
    "dtype": str(BACKBONE_DTYPE), "prompt_version": PROMPT_VERSION,
    "max_length": MAX_LENGTH,
    "prompt_hash": hashlib.sha256(canonical(prompts).encode()).hexdigest()}
cache_key = hashlib.sha256(canonical(cache_meta).encode()).hexdigest()[:16]
cache_path = RUN_DIR / f"qwen_features_{cache_key}.npz"
print("Embedding cache:", cache_path)


Embedding cache: /content/artifacts/jev_synthetic_demo/embedding_cache/qwen_features_5ddec3751f5f14f1.npz


In [ ]:
if cache_path.is_file():
    with np.load(cache_path, allow_pickle=False) as cached:
        if "cache_meta_json" not in cached.files or str(cached["cache_meta_json"].item()) != canonical(cache_meta):
            raise ValueError(f"Cache identity mismatch: remove only {cache_path} and rerun extraction")
        embeddings = cached["embeddings"]
    print("Reused matching feature cache:", cache_path)
else:
    embeddings = extract_embeddings(feature_model, tokenizer, prompts, EXTRACT_BATCH_SIZE)
    expected_shape = (len(data), int(feature_model.backbone.config.hidden_size))
    if embeddings.shape != expected_shape or not np.isfinite(embeddings).all():
        raise ValueError("Extracted embeddings have an invalid shape or non-finite values")
    np.savez_compressed(cache_path, embeddings=embeddings,
                        cache_meta_json=np.array(canonical(cache_meta)))
expected_shape = (len(data), int(feature_model.backbone.config.hidden_size))
if embeddings.shape != expected_shape or not np.isfinite(embeddings).all():
    raise ValueError("Invalid feature cache: dataset size or backbone dimensions differ")


Qwen features: 16/3600
Qwen features: 416/3600
Qwen features: 816/3600
Qwen features: 1216/3600
Qwen features: 1616/3600
Qwen features: 2016/3600
Qwen features: 2416/3600
Qwen features: 2816/3600
Qwen features: 3216/3600


In [ ]:
import gc

HIDDEN_SIZE = embeddings.shape[1]
# Head epochs require only the small cached array. Reload the backbone for inference later.
del feature_model
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()
gc.collect()

In [ ]:
# tokenizer, feature_model, RESOLVED_MODEL_REVISION = build_backbone()
# num_head = sum(p.numel() for p in feature_model.score.parameters())
# print("Trainable head parameters:",num_head, "| frozen backbone:",sum(p.numel() for p in feature_model.backbone.parameters()))
# lengths = [len(ids) for ids in tokenizer(prompts, truncation=False)["input_ids"]]
# print("Prompt tokens median/max:",int(np.median(lengths)),max(lengths))
# if max(lengths) > MAX_LENGTH: raise ValueError("Increase MAX_LENGTH and rerun before extraction")
# cache_meta = {"model_id":MODEL_ID,"revision":RESOLVED_MODEL_REVISION,
#     "dtype":str(BACKBONE_DTYPE),"prompt_version":PROMPT_VERSION,"max_length":MAX_LENGTH,
#     "prompt_hash":hashlib.sha256(canonical(prompts).encode()).hexdigest()}
# cache_key = hashlib.sha256(canonical(cache_meta).encode()).hexdigest()[:16]
# cache_path = RUN_DIR / f"qwen_features_{cache_key}.npz"
# if cache_path.is_file():
#     with np.load(cache_path,allow_pickle=False) as cached:
#         embeddings = cached["embeddings"]
#     print("Reused feature cache:",cache_path)
# else:
#     embeddings = extract_embeddings(feature_model,tokenizer,prompts,EXTRACT_BATCH_SIZE)
#     np.savez_compressed(cache_path,embeddings=embeddings)
# if embeddings.shape[0] != len(data) or not np.isfinite(embeddings).all():
#     raise ValueError("Invalid feature cache")
# HIDDEN_SIZE = embeddings.shape[1]
# # Head epochs require only the small cached array. Reload the backbone for inference later.
# del feature_model
# gc.collect()
# if DEVICE.type == "cuda": torch.cuda.empty_cache()


## 7. Train only the head
Embedding standardization is fitted on the training split only and saved for inference. Cross-entropy uses the observed training distribution with no oversampling/class weights, so probability interpretation is not silently changed by artificial rebalancing. The synthetic sample is not a rare-event prevalence estimate.


In [ ]:
# MAX_LENGTH = 768
# EXTRACT_BATCH_SIZE = 16
# HEAD_EPOCHS = 60
# SPLIT_MODE = "chronological"
# RESEARCH_THRESHOLD_72H = 0.50

In [ ]:
head, embedding_mean, embedding_scale, logits, history = fit_head(embeddings,data,HEAD_EPOCHS)
history.to_csv(ARTIFACT_DIR / "head_training_history.csv",index=False)
history.set_index("epoch")[["train_nll","validation_nll"]].plot(figsize=(7,3),ylabel="Negative log likelihood")
plt.title("Synthetic demo: train and validation loss" if IS_SYNTHETIC_RUN else "Train and validation loss")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "head_train_validation_loss.png", dpi=150)
plt.show()

## 8. Fit temperature on the separate calibration split
A single positive temperature changes probability sharpness without changing the most likely class. It is selected using calibration NLL, never the test split. Temperature fitting improves neither missing physical information nor inadequate event labels; calibration on toy examples is not calibration on real GLOFs.


In [ ]:
calibration_mask = data.split.eq("calibration").to_numpy()
temperature,calibration_nll = fit_temperature(logits[calibration_mask],data.loc[calibration_mask,"target_id"].to_numpy())
print("Temperature:",round(temperature,4), "| calibration NLL:",round(calibration_nll,4))

#### 9. Held-out evaluation and a numeric baseline
Report probability quality and rare-event errors, not training accuracy alone. The numeric baseline consumes exactly the same feature allowlist, with missing-value handling fitted only on training data. It is deliberately included because a Qwen embedding classifier may be less effective than tabular ML for your numeric measurements.


In [ ]:
train_mask = data.split.eq("train").to_numpy()
test_mask = data.split.eq("test").to_numpy()
numeric_baseline = make_pipeline(SimpleImputer(strategy="median",add_indicator=True),
    HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=15,min_samples_leaf=10,random_state=SEED))
numeric_baseline.fit(data.loc[train_mask,FEATURE_COLUMNS],data.loc[train_mask,"target_id"])
numeric_probs = numeric_baseline.predict_proba(data.loc[test_mask,FEATURE_COLUMNS])
assert list(numeric_baseline.classes_) == [0,1,2]
test_y = data.loc[test_mask,"target_id"].to_numpy()
raw_probs = softmax_np(logits[test_mask])
calibrated_probs = softmax_np(logits[test_mask],temperature)
reports = {"synthetic_demo_only":IS_SYNTHETIC_RUN,
    "qwen_head_raw":metrics_report(test_y,raw_probs,RESEARCH_THRESHOLD_72H),
    "qwen_head_temperature_scaled":metrics_report(test_y,calibrated_probs,RESEARCH_THRESHOLD_72H),
    "numeric_baseline":metrics_report(test_y,numeric_probs,RESEARCH_THRESHOLD_72H)}
display(reports)
(ARTIFACT_DIR / "heldout_metrics.json").write_text(json.dumps(clean(reports),indent=2),encoding="utf-8")
test_out = data.loc[test_mask,["sample_id","lake_id","episode_id","assessed_at","target_name"]].copy()
test_out["p_new_onset_24h"] = calibrated_probs[:,1]
test_out["p_new_onset_72h"] = calibrated_probs[:,1]+calibrated_probs[:,2]
test_out["synthetic_demo_only"] = IS_SYNTHETIC_RUN
test_out.to_csv(ARTIFACT_DIR / "heldout_predictions.csv",index=False)
score = calibrated_probs[:,1]+calibrated_probs[:,2]
truth = (test_y>0).astype(int)
bins = pd.cut(score,np.linspace(0,1,6),include_lowest=True)
reliability = pd.DataFrame({"bin":bins,"score":score,"event":truth}).groupby("bin",observed=True).agg(
    mean_score=("score","mean"),event_fraction=("event","mean"),n=("event","size"))
display(reliability)
fig,ax = plt.subplots(figsize=(4,4))
ax.plot([0,1],[0,1],"--",color="grey")
ax.plot(reliability.mean_score,reliability.event_fraction,"o-")
ax.set(xlim=(0,1),ylim=(0,1),xlabel="Mean predicted next-72h probability",ylabel="Observed fraction",
    title="SYNTHETIC held-out examples" if IS_SYNTHETIC_RUN else "Held-out reliability")
plt.tight_layout()
plt.show()

#### 10. Save a reloadable head and preprocessing metadata
This saves the small head, train-only normalization, temperature and exact backbone identity. The backbone must remain in the Hugging Face cache or a local weights folder; a head file alone does not contain Qwen. Only load tensor checkpoints you produced or trust. The snapshot metadata records versions and source/data hashes for reproducibility.


In [ ]:
checkpoint_path = ARTIFACT_DIR / "glof_jev_head.pt"
torch.save({"score_state":head.state_dict(),"feature_mean":torch.from_numpy(embedding_mean),
    "feature_scale":torch.from_numpy(embedding_scale)},checkpoint_path)
metadata = {"model_id":MODEL_ID,"model_revision":RESOLVED_MODEL_REVISION,
    "hidden_size":int(HIDDEN_SIZE),"class_names":list(CLASS_NAMES),"feature_columns":FEATURE_COLUMNS,
    "temperature":temperature,"prompt_version":PROMPT_VERSION,"max_length":MAX_LENGTH,
    "split_mode":SPLIT_MODE,"embedding_cache_key":cache_key,
    "synthetic_demo_only":IS_SYNTHETIC_RUN,"head_training_rows":int(train_mask.sum()),
    "data_sha256":hashlib.sha256(DATA_CSV.read_bytes()).hexdigest(),"seed":SEED,
    "dataset_kind":"SYNTHETIC_SOFTWARE_DEMO" if IS_SYNTHETIC_RUN else "VERIFIED_RESEARCH_DATA",
    "versions":{"torch":torch.__version__,"python":platform.python_version(),
        "transformers":__import__("transformers").__version__},
    "target_definition":"FIRST new onset strictly after assessment: <=24h, (24,72]h, or verified no onset <=72h"}
(ARTIFACT_DIR / "glof_jev_metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
print("Saved head:",checkpoint_path)
print("Saved metadata:",ARTIFACT_DIR / "glof_jev_metadata.json")

#### Save/reload check

Select a held-out row with sufficient image-change measurements. It is evaluated at its fictional assessment clock, preserving historical timestamps rather than relabeling old features as current.


In [ ]:
inference_tokenizer,inference_model,inference_metadata = load_trained_head(ARTIFACT_DIR)
valid_rows = [r for r in data[data.split=="test"].to_dict("records")
              if not measurement_gate(r,r["assessed_at"])]
if not valid_rows: raise ValueError("No held-out row passes measurement quality checks")
demo_row=valid_rows[0]
demo_prediction=predict_glof([demo_row],inference_tokenizer,inference_model,inference_metadata,now=demo_row["assessed_at"])
display(demo_prediction)
a=demo_prediction[0]["answers"]
assert 0 <= a["glof_next_24h"]["noul"] <= a["glof_next_72h"]["noul"] <= 1
del inference_model
gc.collect()
if DEVICE.type=="cuda": torch.cuda.empty_cache()
print("Reload/horizon-consistency check passed")


#### Optional: save the full Qwen backbone for a different/offline host

The head alone doesn't contain Qwen weights. Normally the host reuses its Hugging Face cache or downloads the exact recorded revision. To make the artifacts self-contained, enable this cell, then upload the resulting `backbone/` directory too. Expect several GB of additional download/storage; the supplied ZIP contains code and sample data only.


In [52]:
SAVE_FULL_BACKBONE = False
if SAVE_FULL_BACKBONE:
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=MODEL_ID,revision=RESOLVED_MODEL_REVISION,
                      local_dir=str(ARTIFACT_DIR/"backbone"),
                      allow_patterns=["*.json","*.safetensors","*.txt","*.model","LICENSE","README.md"])
    (ARTIFACT_DIR / "backbone_identity.json").write_text(
        json.dumps({"model_id":MODEL_ID,"revision":RESOLVED_MODEL_REVISION}, indent=2),
        encoding="utf-8")

In [ ]:
# UPLOAD_QWEN_ARTIFACTS = True
# if UPLOAD_QWEN_ARTIFACTS:
#     from glofwatch.aws import upload_tree
#     print("Uploaded files:",len(upload_tree(cfg,ARTIFACT_DIR,cfg.s3_prefix+"/models/qwen_jev")))
# else: print("Artifacts retained locally:",ARTIFACT_DIR)